# Chapter 5 | Probabilistic Modelling of Learning Behaviour

**Companion notebook** · Assessment data · Bayesian network structure learning · Probabilistic inference · Educational interpretation

This notebook consolidates the supplied chapter code into one reproducible sequence. It covers exploratory analysis, discretisation, mutual information, Bayesian Information Criterion (BIC) structure search, conditional probability tables, posterior inference and model interpretation. Figures **5.2–5.13** follow the numbering in the book and are exported as PDF and PNG. Figure 5.1 is the separate conceptual framework.

**Dataset:** Mahmud, K. (2025), *Mendeley Data*, Version 4. **DOI:** [10.17632/vzfyk22fhn.4](https://doi.org/10.17632/vzfyk22fhn.4). **Licence:** CC BY 4.0. Download `Student Performance Dataset 2024.csv` from the [dataset record](https://data.mendeley.com/datasets/vzfyk22fhn/4) and place it alongside this notebook or in a `data/` folder.

**Interpretation:** Learned edges, associations and posterior probabilities describe patterns in observational assessment data; they do not, on their own, identify causal effects. The supplied category may incorporate final-course performance and is not used as an advance predictor in this notebook.

**Dependencies:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`, `networkx`, `IPython` (standard Jupyter scientific environment). The five-variable Bayesian network is estimated with a self-contained discrete BIC hill-climbing implementation, avoiding dependency on a particular third-party graphical-model API.

## 1. Dataset and descriptive analysis

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Libraries and export settings</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Load statistical and plotting packages and define one figure-saving convention.</div>
</div>

In [ ]:
from pathlib import Path
from itertools import product
from functools import lru_cache

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.metrics import mutual_info_score

RANDOM_STATE = 42
OUTPUT_DIR = Path('figures')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300,
                     'axes.titlesize': 13, 'axes.labelsize': 11,
                     'pdf.fonttype': 42, 'ps.fonttype': 42})
COLORS = sns.color_palette('mako', 8)


def save_figure(fig, number, slug):
    """Save each book-numbered figure as a vector PDF and a high-resolution PNG."""
    stem = f'Figure_5_{number:02d}_{slug}'
    fig.savefig(OUTPUT_DIR / f'{stem}.pdf', bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / f'{stem}.png', bbox_inches='tight', dpi=300)
    plt.show()

print('Figure output:', OUTPUT_DIR.resolve())

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Load the published dataset</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Read the original CSV and exclude any student identifier from the analytical table.</div>
</div>

In [ ]:
DATASET_URL = 'https://data.mendeley.com/datasets/vzfyk22fhn/4'
DATASET_FILE = 'Student Performance Dataset 2024.csv'
candidates = [Path(DATASET_FILE), Path('data') / DATASET_FILE,
              Path('datasets') / DATASET_FILE]
csv_path = next((path for path in candidates if path.is_file()), None)
if csv_path is None:
    raise FileNotFoundError(
        f'Download {DATASET_FILE!r} from {DATASET_URL} and place it beside '
        'this notebook or in the data/ directory.'
    )
raw_df = pd.read_csv(csv_path)
df = raw_df.drop(columns=['Student Id', 'Student_Id', 'Student ID'], errors='ignore').copy()
print(f'Data source: {csv_path} | Records: {len(df)} | Variables: {df.shape[1]}')
display(df.head())

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Validate assessment variables</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Check required fields, numerical data, missing values and repeated records before modelling.</div>
</div>

In [ ]:
SCORE_COLUMNS = ['Quiz', 'Midterm', 'Assignment_1', 'Assignment_2',
                 'Assignment_3', 'Project', 'Presentation',
                 'Final_Exam_Marks', 'Total']
REQUIRED_COLUMNS = SCORE_COLUMNS + ['Categories']
missing_fields = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_fields:
    raise ValueError('Required dataset columns are missing: ' + ', '.join(missing_fields))
for column in SCORE_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df['Categories'] = df['Categories'].astype('string').str.strip()
missing_report = df[REQUIRED_COLUMNS].isna().sum()
print(f'Missing values: {int(missing_report.sum())}; duplicate complete records: '
      f'{int(df[REQUIRED_COLUMNS].duplicated().sum())}')
if missing_report.any():
    display(missing_report[missing_report > 0].to_frame('Missing values'))
    raise ValueError('Resolve missing assessment values before running the analysis.')
if not len(df):
    raise ValueError('The dataset is empty.')
print('Performance categories:', df['Categories'].value_counts().to_dict())
display(df[SCORE_COLUMNS].describe().round(2))

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.2 | Performance categories</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Show the observed distribution of the published course-performance categories.</div>
</div>

In [ ]:
preferred_order = ['Fail', 'Pass', 'Distinction', 'Excellent', 'Exceptional']
category_order = [x for x in preferred_order if x in set(df['Categories'])]
category_order += sorted(set(df['Categories']) - set(category_order))
counts = df['Categories'].value_counts().reindex(category_order, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.bar(counts.index, counts.values, color=sns.color_palette('Blues', len(counts)))
ax.set(xlabel='Performance category', ylabel='Number of records',
       title='Distribution of Student Performance Categories')
for i, val in enumerate(counts.values):
    ax.text(i, val, str(int(val)), ha='center', va='bottom', fontsize=9)
fig.tight_layout()
save_figure(fig, 2, 'performance_categories')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.3 | Final examination scores</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Display the distribution of final examination marks without inferring an underlying probability law.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.6))
sns.histplot(df['Final_Exam_Marks'], bins=16, kde=True, color=COLORS[4], ax=ax)
ax.set(xlabel='Final examination marks', ylabel='Frequency',
       title='Distribution of Final Examination Marks')
fig.tight_layout()
save_figure(fig, 3, 'final_exam_distribution')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.4 | Quiz and midterm relationships</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Compare midterm and total scores, with quiz results represented by marker size.</div>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.4))
sns.scatterplot(data=df, x='Midterm', y='Total', hue='Categories',
                hue_order=category_order, size='Quiz', sizes=(25, 150),
                alpha=0.65, ax=ax)
ax.set(xlabel='Midterm score', ylabel='Total course score',
       title='Midterm, Quiz and Total Course Scores')
ax.legend(title='Category / quiz', bbox_to_anchor=(1.02, 1),
          loc='upper left', fontsize=8)
fig.tight_layout()
save_figure(fig, 4, 'midterm_quiz_total')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.5 | Assignment profiles</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Examine differences in three assignment scores across performance categories.</div>
</div>

In [ ]:
long_assignments = df.melt(id_vars='Categories',
    value_vars=['Assignment_1', 'Assignment_2', 'Assignment_3'],
    var_name='Assessment', value_name='Score')
fig, ax = plt.subplots(figsize=(9, 5.0))
sns.boxplot(data=long_assignments, x='Assessment', y='Score',
            hue='Categories', hue_order=category_order, ax=ax)
ax.set(title='Assignment Performance Across Categories', xlabel='Assignment')
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.tight_layout()
save_figure(fig, 5, 'assignment_performance')

## 2. Discretisation and dependency analysis

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Prepare discrete assessment states</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Group examination and combined assignment scores into low, medium and high states.</div>
</div>

In [ ]:
STATES = ['L', 'M', 'H']


def terciles(values, name):
    series = pd.to_numeric(values)
    try:
        result = pd.qcut(series, q=3, labels=STATES)
    except ValueError:
        if series.nunique() < 3:
            raise ValueError(f'{name} has fewer than three distinct values.')
        print(f'Note: tied quantiles in {name}; using stable rank-based terciles.')
        result = pd.qcut(series.rank(method='first'), q=3, labels=STATES)
    return pd.Series(result.astype(str), index=series.index, name=name)


discrete = pd.DataFrame(index=df.index)
discrete['Quiz'] = terciles(df['Quiz'], 'Quiz')
discrete['Midterm'] = terciles(df['Midterm'], 'Midterm')
discrete['Assignments'] = terciles(
    df[['Assignment_1', 'Assignment_2', 'Assignment_3']].sum(axis=1), 'Assignments')
discrete['Final'] = terciles(df['Final_Exam_Marks'], 'Final')
discrete['Category'] = df['Categories'].astype(str)
assert discrete.notna().all().all()
display(discrete.head())
print('State counts:')
display(discrete.apply(lambda col: col.value_counts()).fillna(0).astype(int))

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.6 | Mutual information</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Measure the reduction in category uncertainty associated with each discretised assessment.</div>
</div>

In [ ]:
core_features = ['Quiz', 'Midterm', 'Assignments', 'Final']
mi_category = pd.Series({col: mutual_info_score(discrete[col], discrete['Category']) /
                              np.log(2) for col in core_features}).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(mi_category.index, mi_category.values, color=COLORS[3])
ax.set(xlabel='Mutual information with category (bits)',
       title='Information Gain Across Assessment Components')
fig.tight_layout()
save_figure(fig, 6, 'mutual_information')
print('Note: mutual information indicates association, not causation.')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.7 | Dependency matrix</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Calculate Pearson correlations between the original numerical assessment variables.</div>
</div>

In [ ]:
correlation = df[SCORE_COLUMNS].corr()
fig, ax = plt.subplots(figsize=(10.5, 8.5))
mask = np.triu(np.ones(correlation.shape, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f',
            cmap='BrBG', center=0, vmin=-1, vmax=1, square=True,
            linewidths=0.4, cbar_kws={'shrink': 0.75}, ax=ax)
ax.set(title='Dependency Matrix for Numerical Assessment Variables')
fig.tight_layout()
save_figure(fig, 7, 'dependency_matrix')

## 3. Bayesian network structure and parameter estimation

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Discrete BIC structure search</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Implement reproducible hill climbing over directed acyclic graphs using a penalised likelihood score.</div>
</div>

In [ ]:
def learn_bic_graph(data, max_parents=2, max_iterations=80, permitted=None):
    """Score-based structure learning (BIC); edges are observational, not causal."""
    data = data.astype(str).copy()
    nodes = list(data.columns)
    n = len(data)
    cardinality = {v: data[v].nunique() for v in nodes}

    @lru_cache(None)
    def local_score(target, parent_tuple):
        parents = list(parent_tuple)
        if parents:
            cells = data.groupby(parents + [target], observed=True).size()
            parent_totals = data.groupby(parents, observed=True).size()
            totals = parent_totals.reindex(cells.index.droplevel(-1)).to_numpy()
            counts = cells.to_numpy()
            parent_configs = int(np.prod([cardinality[p] for p in parents]))
        else:
            counts = data[target].value_counts().to_numpy()
            totals = np.full(len(counts), n)
            parent_configs = 1
        log_likelihood = np.sum(counts * np.log(counts / totals))
        free_parameters = (cardinality[target] - 1) * parent_configs
        return float(log_likelihood - 0.5 * free_parameters * np.log(n))

    def score(graph):
        return sum(local_score(v, tuple(sorted(graph.predecessors(v)))) for v in nodes)

    current = nx.DiGraph()
    current.add_nodes_from(nodes)
    current_score = score(current)
    operations = 0
    for _ in range(max_iterations):
        best_gain, best_graph = 1e-6, None
        # Try adding, removing, or reversing each directed edge.
        for u in nodes:
            for v in nodes:
                if u == v:
                    continue
                options = []
                if current.has_edge(u, v):
                    candidate = current.copy(); candidate.remove_edge(u, v)
                    options.append(candidate)
                    if (permitted is None or permitted(v, u)):
                        reversed_graph = candidate.copy()
                        if reversed_graph.in_degree(u) < max_parents:
                            reversed_graph.add_edge(v, u)
                            options.append(reversed_graph)
                elif not current.has_edge(v, u) and (permitted is None or permitted(u, v)):
                    if current.in_degree(v) < max_parents:
                        candidate = current.copy(); candidate.add_edge(u, v)
                        options.append(candidate)
                for candidate in options:
                    if not nx.is_directed_acyclic_graph(candidate):
                        continue
                    gain = score(candidate) - current_score
                    if gain > best_gain:
                        best_gain, best_graph = gain, candidate
        if best_graph is None:
            break
        current = best_graph
        current_score += best_gain
        operations += 1
    return current, current_score, operations

print('BIC hill-climbing functions ready.')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.8 | Learned network structure</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Estimate the five-variable dependency graph and plot the BIC-selected directed edges.</div>
</div>

In [ ]:
core_graph, core_bic, core_steps = learn_bic_graph(discrete, max_parents=2)
print(f'BIC score: {core_bic:.2f}; accepted search steps: {core_steps}')
print('Learned directed dependencies:', sorted(core_graph.edges()))
fig, ax = plt.subplots(figsize=(9.2, 5.8))
pos = nx.spring_layout(core_graph, seed=RANDOM_STATE)
nx.draw_networkx_nodes(core_graph, pos, node_size=5000,
                       node_color='#E8F0F5', edgecolors='#324A5F', ax=ax)
nx.draw_networkx_labels(core_graph, pos, font_size=10, font_color='#173244', font_weight='bold', ax=ax)
nx.draw_networkx_edges(core_graph, pos, arrows=True, arrowsize=21,
                       width=1.7, node_size=5000, ax=ax)
ax.set(title='BIC-Selected Bayesian Network Structure')
ax.set_xlim(min(p[0] for p in pos.values())-0.30, max(p[0] for p in pos.values())+0.30)
ax.set_ylim(min(p[1] for p in pos.values())-0.30, max(p[1] for p in pos.values())+0.30)
ax.axis('off'); fig.tight_layout()
save_figure(fig, 8, 'learned_bayesian_structure')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Estimate conditional probability tables</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Fit smoothed discrete probability tables for every node in the learned network.</div>
</div>

In [ ]:
def fit_discrete_cpds(data, graph, prior=0.5):
    """Estimate P(node | parents) with a symmetric Dirichlet prior."""
    state_space = {name: sorted(data[name].astype(str).unique()) for name in graph.nodes()}
    cpds = {}
    for node in graph.nodes():
        parents = sorted(graph.predecessors(node))
        table = {}
        configurations = product(*(state_space[p] for p in parents)) if parents else [()]
        for config in configurations:
            mask = np.ones(len(data), dtype=bool)
            for parent, state in zip(parents, config):
                mask &= (data[parent].to_numpy() == state)
            counts = data.loc[mask, node].value_counts().reindex(
                state_space[node], fill_value=0).to_numpy(dtype=float)
            probs = (counts + prior) / (counts.sum() + prior * len(counts))
            table[tuple(config)] = dict(zip(state_space[node], probs))
        cpds[node] = {'parents': parents, 'table': table}
    return state_space, cpds

state_space, cpds = fit_discrete_cpds(discrete, core_graph)
print('Conditional probability tables fitted for:', ', '.join(cpds))

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Table 5.1 | Final marks by category</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Tabulate observed final-score states conditional on course-performance categories.</div>
</div>

In [ ]:
category_columns = [item for item in
    ['Distinction', 'Excellent', 'Exceptional', 'Fail', 'Pass']
    if item in state_space['Category']]
category_columns += [x for x in state_space['Category'] if x not in category_columns]
conditional_final = pd.crosstab(discrete['Final'], discrete['Category'],
                                normalize='columns').reindex(
    index=STATES, columns=category_columns, fill_value=0)
conditional_final.index.name = 'Final state'
display(conditional_final.round(4))
conditional_final.to_csv(OUTPUT_DIR / 'Table_5_1_Final_given_Category.csv')
print('This is an empirical conditional table; it is not necessarily the learned '
      'network CPD because the learned graph may have different parents.')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.9 | Augmented design illustration</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Display the hypothesised assessment dependency structure as a conceptual example, not a fitted network.</div>
</div>

In [ ]:
concept_graph = nx.DiGraph()
concept_graph.add_edges_from([
    ('Category','Assignments'), ('Category','Quiz'),
    ('Category','Midterm'), ('Category','Final'),
    ('Assignments','Quiz'), ('Quiz','Midterm'), ('Midterm','Final')
])
positions = {'Category':(0.2,1.2), 'Assignments':(1.2,0.1),
             'Quiz':(2.4,0.2), 'Midterm':(3.6,0.4), 'Final':(4.8,0.6)}
fig, ax = plt.subplots(figsize=(10, 4.7))
nx.draw_networkx_nodes(concept_graph, positions, node_color='#E8F0F5',
                       node_size=6200, edgecolors='#324A5F', ax=ax)
nx.draw_networkx_labels(concept_graph, positions, font_size=10, font_color='#173244', font_weight='bold', ax=ax)
nx.draw_networkx_edges(concept_graph, positions, arrows=True, node_size=6200,
                       arrowsize=17, width=1.5, connectionstyle='arc3,rad=0.06', ax=ax)
ax.set(title='Illustrative Augmented Bayesian Architecture (Not Fitted)')
ax.set_xlim(-0.55, 5.55); ax.set_ylim(-0.60, 2.00)
ax.axis('off'); fig.tight_layout()
save_figure(fig, 9, 'augmented_illustrative_architecture')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.10 | Extended assessment network</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Learn a data-driven nine-node network with an explicit ordering assumption for the assessment stages.</div>
</div>

In [ ]:
extended = pd.DataFrame(index=df.index)
for var in ['Assignment_1','Assignment_2','Assignment_3',
            'Quiz','Project','Presentation','Midterm','Final_Exam_Marks']:
    extended[var] = terciles(df[var], var)
extended['Categories'] = df['Categories'].astype(str)

# Ordering is a modelling assumption, not independently validated timing.
layer = {'Assignment_1':0, 'Assignment_2':0, 'Assignment_3':0,
         'Project':0, 'Presentation':0, 'Quiz':1,
         'Midterm':2, 'Final_Exam_Marks':3, 'Categories':4}


def permitted_extended(source, target):
    return layer[source] < layer[target]

extended_graph, extended_bic, _ = learn_bic_graph(
    extended, max_parents=2, max_iterations=65,
    permitted=permitted_extended)
print('Extended learned edges:', sorted(extended_graph.edges()))
columns = {0:0.1,1:2.0,2:3.9,3:5.8,4:7.7}
positions = {}
for stage in sorted(set(layer.values())):
    members = sorted([v for v in extended.columns if layer[v] == stage])
    for j, member in enumerate(members):
        positions[member] = (columns[stage], 1.4*(j-(len(members)-1)/2))
labels = {v: v.replace('Assignment_', 'Assignment\n').replace('Final_Exam_Marks', 'Final\nExam\nMarks') for v in extended.columns}
fig, ax = plt.subplots(figsize=(12.5, 7.0))
nx.draw_networkx_nodes(extended_graph, positions, node_color='#E8F0F5',
                       node_size=5200, edgecolors='#324A5F', ax=ax)
nx.draw_networkx_labels(extended_graph, positions, labels=labels,
                        font_size=8.5, font_color='#173244', font_weight='bold', ax=ax)
nx.draw_networkx_edges(extended_graph, positions, arrows=True, arrowsize=16,
                       node_size=5200, width=1.5, ax=ax)
ax.set(title='BIC-Selected Dependencies with Assumed Assessment Ordering')
ax.set_xlim(-0.7, 8.5); ax.set_ylim(-3.55, 3.55)
ax.axis('off'); fig.tight_layout()
save_figure(fig, 10, 'extended_assessment_network')

## 4. Probabilistic inference and diagnostic interpretation

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Exact posterior inference</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Calculate conditional probabilities by enumerating the small five-variable joint distribution.</div>
</div>

In [ ]:
def query_posterior(target, evidence=None):
    evidence = evidence or {}
    if target not in state_space:
        raise KeyError(f'Unknown query variable: {target}')
    for variable, value in evidence.items():
        if variable not in state_space or value not in state_space[variable]:
            raise ValueError(f'Invalid evidence: {variable}={value}')
    nodes = list(core_graph.nodes())
    weights = dict.fromkeys(state_space[target], 0.0)
    denominator = 0.0
    for values in product(*(state_space[v] for v in nodes)):
        case = dict(zip(nodes, values))
        if any(case[key] != val for key, val in evidence.items()):
            continue
        prob = 1.0
        for node in nodes:
            spec = cpds[node]
            parents = tuple(case[p] for p in spec['parents'])
            prob *= spec['table'][parents][case[node]]
        denominator += prob
        weights[case[target]] += prob
    if denominator <= 0:
        raise ValueError('No positive probability is available for this query.')
    return pd.Series({state: value/denominator for state,value in weights.items()},
                     name=target)

predicted = query_posterior('Final', {'Quiz':'L'})
diagnostic = query_posterior('Assignments', {'Final':'L'})
profile = query_posterior('Category', {'Midterm':'H', 'Assignments':'H'})
print('P(Final | Quiz=L):')
display(predicted.round(3).to_frame('Probability'))
print('P(Category | Midterm=H, Assignments=H):')
display(profile.round(3).to_frame('Probability'))

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.11 | Diagnostic posterior</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Visualise the assignment-state distribution conditional on low final-examination performance.</div>
</div>

In [ ]:
posterior = diagnostic.reindex(STATES).fillna(0)
fig, ax = plt.subplots(figsize=(7.6, 4.7))
bars = ax.bar(posterior.index, posterior.values, color=COLORS[4])
ax.set(xlabel='Combined assignment state', ylabel='Posterior probability',
       ylim=(0, 1.08), title="P(Assignments | Final = Low)")
for bar in bars:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.025,
            f'{bar.get_height():.2f}', ha='center')
fig.tight_layout()
save_figure(fig, 11, 'assignment_posterior_given_low_final')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Conditional evidence comparison</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Compare posterior probabilities with and without additional midterm evidence.</div>
</div>

In [ ]:
baseline = query_posterior('Assignments', {'Final':'L'})
with_midterm = query_posterior('Assignments', {'Final':'L', 'Midterm':'L'})
evidence_comparison = pd.DataFrame({
    'Final=L': baseline, 'Final=L and Midterm=L': with_midterm,
})
display(evidence_comparison.round(4))
print('Differences represent conditional association; explaining away requires '
      'a suitable graph structure and must not be assumed.')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.12 | Statistical dependence across edges</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Replace arbitrary edge weights with measured mutual information for the fitted network.</div>
</div>

In [ ]:
edge_mi = pd.Series({f'{u} → {v}': mutual_info_score(
    discrete[u], discrete[v]) / np.log(2)
    for u, v in core_graph.edges()}, dtype=float).sort_values()
fig, ax = plt.subplots(figsize=(8.2, max(3.3, len(edge_mi)*0.52+1.4)))
if edge_mi.empty:
    ax.text(0.5, 0.5, 'No directed edges selected by BIC',
            transform=ax.transAxes, ha='center', va='center')
    ax.axis('off')
else:
    ax.barh(edge_mi.index, edge_mi.values, color=COLORS[3])
    ax.set(xlabel='Pairwise mutual information (bits)',
           title='Statistical Dependence on Learned Network Edges')
fig.tight_layout()
save_figure(fig, 12, 'measured_edge_dependencies')

<div style="background-color:#F3F7FA;border-left:4px solid #476B82;border-radius:5px;padding:11px 14px;margin:15px 0 7px;">
<strong style="color:#253F50;">Figure 5.13 | Markov blanket of the outcome</strong><div style="font-size:0.94em;color:#485A65;margin-top:3px;">Identify the category node’s Markov blanket and plot measured associations rather than fixed importance values.</div>
</div>

In [ ]:
target = 'Category'
parents = set(core_graph.predecessors(target))
children = set(core_graph.successors(target))
co_parents = {parent for child in children
              for parent in core_graph.predecessors(child)}
markov_blanket = sorted((parents | children | co_parents) - {target})
print('Markov blanket of Category:', markov_blanket)
mb_mi = pd.Series({v: mutual_info_score(discrete[v],discrete[target]) /
                    np.log(2) for v in markov_blanket}, dtype=float).sort_values()
fig, ax = plt.subplots(figsize=(8.2, max(3.0, len(mb_mi)*0.6+1.7)))
if mb_mi.empty:
    ax.text(0.5,0.5,'No Markov blanket variables identified',
            ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')
else:
    ax.barh(mb_mi.index, mb_mi.values, color=COLORS[5])
    ax.set(xlabel='Mutual information with category (bits)',
           title='Markov Blanket and Observed Associations')
fig.tight_layout()
save_figure(fig, 13, 'markov_blanket')

## 5. Interpretation and reproducibility notes

The resulting figures support interpretation of **observed assessment dependencies**. The BIC-selected structures in Figures 5.8 and 5.10 are estimated from the data; Figure 5.9 is an explicitly illustrative architecture. Pairwise correlations and mutual information do not validate causal pathways or the effectiveness of instructional interventions.

**Important methodological boundaries:**

- Performance categories may be constructed from overall course marks; using them to predict earlier outcomes would introduce future-information leakage.
- Tercile thresholds depend on the supplied dataset. Rank-based fallback, if triggered by repeated quantile values, is reported during execution.
- Conditional probability tables provide observational summaries. They do not estimate effects of interventions.
- The data-derived structures and measures can differ from the figures or numerical statements in the printed chapter. Such differences should be investigated rather than replaced with illustrative constants.
- Figure files are written to `figures/`; the published CSV should be obtained directly from its cited source and retained separately.